# ETS MARL — Q-Learning Baseline

This notebook trains and analyses a **tabular Q-learning** baseline for the EU ETS multi-agent environment, and compares it against the default-config PPO/HAPPO sweep — the comparison is the headline contribution of this notebook to **Sub-RQ 1 (simulation credibility)**.

The Q-learning agents use:
- **State discretisation**: 5 features × 3 bins = 243 discrete states
- **Action profiles**: 6 auction × 4 secondary = 24 predefined strategy templates
- **Standard Q-learning**: $\alpha=0.1$, $\gamma=0.95$, linear $\varepsilon$-decay

Crucially, the baseline trains in the **same `ETSEnvironment`** as the PPO trainer — same warm-start, same uniform-price auction, same MSR / penalty / ESG mechanisms — so reward, clearing price, compliance rate, and the anchor-invariant `quality_score` are directly comparable.

**Notebook layout.** §1–§5 setup and training. §6–§8 standalone Q-learning analysis (training curves, Q-table inspection, strategy-frequency). §9 greedy evaluation. §10 single-seed Q-learning vs PPO overlay. §11 **cross-seed default-sweep comparison (Sub-RQ 1)**. §13 takeaways and takeaways.


## 1. Setup

In [ ]:
import os, sys

# Navigate to project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

print(f'Working directory: {os.getcwd()}')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import time

from src.environment.ets_environment import ETSEnvironment
from src.agents.q_learning_agent import QLearningAgent, StateDiscretizer, ActionProfileMapper
from src.train_qlearning import train_qlearning, evaluate_qlearning, merge_configs
from src.analysis.qlearning_analysis import (
    plot_qtable_heatmaps, plot_secondary_heatmaps,
    plot_strategy_frequency, plot_reward_comparison,
    plot_price_comparison, plot_green_comparison,
    plot_compliance_comparison, load_training_log
)

plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12, 5)

print('All imports successful.')

## 2. Load Configuration

In [ ]:
with open('configs/default.yaml') as f:
    base_config = yaml.safe_load(f)

with open('configs/qlearning.yaml') as f:
    ql_config = yaml.safe_load(f)

config = merge_configs(base_config, ql_config)

n_agents = config['companies']['n_agents']
n_years = config['simulation']['n_years']
ql_params = config['qlearning']

print(f'Agents: {n_agents} learning + {config["companies"].get("n_bot_agents", 0)} bots')
print(f'Episode length: {n_years} years')
print(f'Q-learning: {ql_params["n_episodes"]} episodes, '
      f'alpha={ql_params["alpha"]}, gamma={ql_params["gamma"]}')
print(f'Epsilon: {ql_params["epsilon_start"]} -> {ql_params["epsilon_end"]} '
      f'over {ql_params["epsilon_decay_frac"]*100:.0f}% of training')
print(f'Reward shaping: beta={config["reward"]["shaping_beta"]}, '
      f'gamma={config["reward"]["shaping_gamma"]} '
      f'(same reward structure as PPO)')

## 3. Explore the State/Action Space

Before training, let's understand how the discretizer and action profiles work.

In [ ]:
# Show state space structure
disc = StateDiscretizer()
print(f'Total discrete states: {disc.N_STATES}')
print(f'Features and bin edges:')
for name, edges in disc._BINS.items():
    print(f'  {name:10s}: low <= {edges[0]:.2f} | mid <= {edges[1]:.2f} | high')

print(f'\nAction profiles:')
mapper = ActionProfileMapper()
print(f'  Auction ({mapper.N_AUCTION_PROFILES}): {mapper.auction_profile_names()}')
print(f'  Secondary ({mapper.N_SECONDARY_PROFILES}): {mapper.secondary_profile_names()}')
print(f'  Total action combinations: {mapper.N_AUCTION_PROFILES * mapper.N_SECONDARY_PROFILES}')

In [ ]:
# Show what each auction profile produces at a reference price
env_temp = ETSEnvironment(config, seed=42)
env_temp.reset(seed=42)

ref_price = 80.0  # reference MA3 price
company = env_temp.companies[0]  # coal-heavy agent

print(f'Auction profiles at MA3 price = {ref_price} EUR/t (Agent A1: coal-heavy):')
print(f'{"Profile":<15s} {"Bid":>7s} {"Qty":>5s} {"Inv%":>5s} {"Tech":>10s}')
print('-' * 45)
for p_idx in range(6):
    action = mapper.get_auction_action(p_idx, company, ref_price, config)
    tech_idx = np.argmax(action[3:6])
    tech_names = ['onshore', 'offshore', 'solar']
    name = mapper.auction_profile_names()[p_idx]
    print(f'{name:<15s} {action[0]:7.1f} {action[1]:5.2f} {action[2]*100:5.1f} {tech_names[tech_idx]:>10s}')

## 4. Run a Random-Policy Episode (Pre-training Baseline)

In [ ]:
# Run one episode with fully random actions to establish a floor
env = ETSEnvironment(config, seed=42)
agents_random = [QLearningAgent(i, seed=42+i) for i in range(n_agents)]

obs1, _ = env.reset(seed=42)
total_rewards = np.zeros(n_agents)
prices, green_fracs = [], []

for year in range(n_years):
    price_ma3 = env._compute_price_ma3()
    
    # Phase 1 (random)
    auction_actions = np.zeros((n_agents, 6), dtype=np.float32)
    a1_indices = np.zeros(n_agents, dtype=int)
    for i in range(n_agents):
        action, a1_idx = agents_random[i].select_auction_action(
            obs1[i], env.companies[i], price_ma3, config, epsilon=1.0)
        auction_actions[i] = action
        a1_indices[i] = a1_idx
    
    obs2, _ = env.step_auction(auction_actions)
    
    # Phase 2 (random)
    sec_actions = np.zeros((n_agents, 2), dtype=np.float32)
    for i in range(n_agents):
        action, _ = agents_random[i].select_secondary_action(
            obs2[i], env.companies[i], env._phase1_clearing_price,
            config, a1_idx=a1_indices[i], epsilon=1.0)
        sec_actions[i] = action
    
    obs1, rewards, terminated, _, info = env.step_secondary(sec_actions)
    total_rewards += rewards
    
    yl = info.get('year_log', {})
    prices.append(yl.get('clearing_price', 0))
    green_fracs.append([env.companies[i].green_frac for i in range(n_agents)])

print('Random policy episode results:')
print(f'  Total rewards: {["{:.2f}".format(r) for r in total_rewards]}')
print(f'  Final green fracs: {["{:.1f}%".format(env.companies[i].green_frac*100) for i in range(n_agents)]}')
print(f'  Price range: {min(prices):.1f} - {max(prices):.1f} EUR/t')

In [ ]:
# Plot random episode
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(prices, 'b-o', markersize=4)
axes[0].set_title('Clearing Price')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('EUR/t')
axes[0].grid(True, alpha=0.3)

green_arr = np.array(green_fracs)
for i in range(n_agents):
    axes[1].plot(green_arr[:, i], label=f'A{i+1}', alpha=0.7)
axes[1].set_title('Green Fraction')
axes[1].set_xlabel('Year')
axes[1].legend(fontsize=7, ncol=2)
axes[1].grid(True, alpha=0.3)

axes[2].bar(range(n_agents), total_rewards)
axes[2].set_title('Total Reward')
axes[2].set_xlabel('Agent')
axes[2].set_xticks(range(n_agents))
axes[2].set_xticklabels([f'A{i+1}' for i in range(n_agents)])
axes[2].grid(True, alpha=0.3)

plt.suptitle('Random Policy Baseline (1 Episode)', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Train Q-Learning Agents

Train for 5,000 episodes (adjustable via config). This should take a few minutes.

In [ ]:
SEED = 42

# Run training
trained_agents, trained_config = train_qlearning(base_config, ql_config, seed=SEED)

## 6. Training Curves

In [ ]:
import glob
import re

# Resolve results directory even if Section 5 was run on another device
if 'trained_config' in globals() and isinstance(trained_config, dict):
    results_dir = trained_config.get('logging', {}).get('results_dir', 'results/qlearning/')
else:
    results_dir = 'results/qlearning/'

# Find training logs; prefer the current SEED if available
log_pattern = os.path.join(results_dir, 'ql_training_log_s*.csv')
log_candidates = glob.glob(log_pattern)

if not log_candidates:
    # Fallback: search recursively under results/
    log_candidates = glob.glob(os.path.join('results', '**', 'ql_training_log_s*.csv'), recursive=True)

if not log_candidates:
    raise FileNotFoundError(
        "No Q-learning training log found. Expected files like 'ql_training_log_s42.csv'."
    )

def _seed_from_path(path):
    m = re.search(r'ql_training_log_s(\d+)\.csv$', path.replace('\\', '/'))
    return int(m.group(1)) if m else -1

preferred_seed = globals().get('SEED', None)
if preferred_seed is not None:
    matching = [p for p in log_candidates if _seed_from_path(p) == preferred_seed]
else:
    matching = []

if matching:
    log_path = matching[0]
else:
    # Use latest modified log if preferred seed is unavailable
    log_path = max(log_candidates, key=os.path.getmtime)

ACTIVE_SEED = _seed_from_path(log_path)
SEED = ACTIVE_SEED  # keep downstream cells consistent
results_dir = os.path.dirname(log_path) or results_dir

ql_df = pd.read_csv(log_path)

print(f'Training log loaded: {log_path}')
print(f'Active seed: {ACTIVE_SEED}')
print(f'Training log: {len(ql_df)} episodes')
print(f'Columns: {list(ql_df.columns)[:10]}...')

# Optional artifact: Q-tables
qtable_candidates = []
qtable_pref = os.path.join(results_dir, f'qtables_s{ACTIVE_SEED}_final.npz')
if os.path.exists(qtable_pref):
    qtable_candidates = [qtable_pref]
else:
    qtable_candidates = glob.glob(os.path.join(results_dir, 'qtables_s*_final.npz'))

HAS_QTABLES = len(qtable_candidates) > 0
qtables = None
qtable_path = None
if HAS_QTABLES:
    qtable_path = qtable_candidates[0]
    qtables = dict(np.load(qtable_path, allow_pickle=False))
    print(f'Q-tables loaded: {qtable_path}')
else:
    print('Q-tables not found. Q-table analysis and evaluation cells will be skipped.')

# Build evaluation inputs if possible
EVAL_AGENTS = None
EVAL_CONFIG = None

if 'trained_agents' in globals() and 'trained_config' in globals():
    EVAL_AGENTS = trained_agents
    EVAL_CONFIG = trained_config
    print('Evaluation source: in-memory trained_agents/trained_config')
else:
    # Rebuild config from YAML (works when notebook restarted on another machine)
    if 'config' not in globals():
        with open('configs/default.yaml') as f:
            _base = yaml.safe_load(f)
        with open('configs/qlearning.yaml') as f:
            _ql = yaml.safe_load(f)
        config = merge_configs(_base, _ql)
    EVAL_CONFIG = config

    if HAS_QTABLES:
        n_agents_local = EVAL_CONFIG['companies']['n_agents']
        ql_params_local = EVAL_CONFIG.get('qlearning', {})
        restored_agents = [
            QLearningAgent(
                i,
                alpha=ql_params_local.get('alpha', 0.1),
                gamma=ql_params_local.get('gamma', 0.95),
                seed=ACTIVE_SEED + i,
            )
            for i in range(n_agents_local)
        ]
        for i in range(n_agents_local):
            key = f'agent_{i}'
            if key in qtables:
                restored_agents[i].q_table = qtables[key]
        EVAL_AGENTS = restored_agents
        print('Evaluation source: agents reconstructed from saved Q-tables + YAML config')

ql_df.tail(3)

In [ ]:
# Reward curves
window = 50
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
fig.suptitle(f'Per-Agent Reward (smoothed, window={window})', fontsize=14)

archetypes = ['Coal/Fin', 'Coal/Green', 'Gas/Fin', 'Gas/Green',
              'Trans/Fin', 'Trans/Green', 'Green/Fin', 'Green/Green']

for i in range(n_agents):
    ax = axes[i // 4, i % 4]
    col = f'reward_A{i+1}'
    rewards = ql_df[col].astype(float).values
    smoothed = pd.Series(rewards).rolling(window).mean()
    ax.plot(smoothed, color='tab:blue', alpha=0.8)
    ax.fill_between(range(len(smoothed)),
                    pd.Series(rewards).rolling(window).quantile(0.25),
                    pd.Series(rewards).rolling(window).quantile(0.75),
                    alpha=0.2, color='tab:blue')
    ax.set_title(f'A{i+1}: {archetypes[i]}', fontsize=10)
    ax.set_xlabel('Episode')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Price, green fraction, compliance, and quality_score trajectories
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
axes = axes.ravel()

# Price
prices = ql_df['clearing_price_last'].astype(float)
axes[0].plot(prices.rolling(window).mean(), color='tab:red', alpha=0.8)
axes[0].set_title('Clearing Price (smoothed)')
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('EUR/t')
axes[0].grid(True, alpha=0.3)

# Green fractions per agent
for i in range(n_agents):
    gf = ql_df[f'green_frac_A{i+1}'].astype(float)
    axes[1].plot(gf.rolling(window).mean(), label=f'A{i+1}', alpha=0.7)
axes[1].set_title('Green Fraction (smoothed)')
axes[1].set_xlabel('Episode'); axes[1].legend(fontsize=7, ncol=2)
axes[1].grid(True, alpha=0.3)

# Compliance rate per agent
for i in range(n_agents):
    cr = ql_df[f'compliance_rate_A{i+1}'].astype(float)
    axes[2].plot(cr.rolling(window).mean(), label=f'A{i+1}', alpha=0.7)
axes[2].set_title('Compliance Rate (smoothed)')
axes[2].set_xlabel('Episode'); axes[2].legend(fontsize=7, ncol=2)
axes[2].set_ylim(0, 1.05); axes[2].grid(True, alpha=0.3)

# Quality score (anchor-invariant) — same metric the PPO trainer logs
if 'quality_score' in ql_df.columns:
    qs = pd.to_numeric(ql_df['quality_score'], errors='coerce')
    axes[3].plot(qs.rolling(window).mean(), color='tab:purple', alpha=0.85)
    axes[3].axhline(0, color='grey', lw=0.5, ls='--')
    axes[3].set_title('quality_score  (≥0 = recognisable EU-ETS)')
    axes[3].set_xlabel('Episode'); axes[3].set_ylabel('signed [-5, +5]')
    axes[3].grid(True, alpha=0.3)
else:
    axes[3].set_axis_off()
    axes[3].text(0.5, 0.5, 'quality_score not in CSV\n(re-run training)',
                 ha='center', va='center', transform=axes[3].transAxes)

plt.tight_layout(); plt.show()


## 7. Q-Table Analysis

In [ ]:
# Load final Q-tables (already loaded in Cell 15)
if HAS_QTABLES and qtables is not None:
    print(f'Loaded Q-tables for {len(qtables)} agents from: {qtable_path}')
    for key, qt in qtables.items():
        nonzero = np.count_nonzero(qt)
        total = qt.size
        print(f'  {key}: shape={qt.shape}, '
              f'non-zero={nonzero}/{total} ({nonzero/total*100:.1f}%), '
              f'Q range=[{qt.min():.3f}, {qt.max():.3f}]')
else:
    print('No Q-tables available. Skipping Section 7 plots and top-Q analysis.')

In [ ]:
# Q-table heatmaps: dominant auction profile per state
if HAS_QTABLES and qtables is not None:
    plot_qtable_heatmaps(qtables, n_agents)
    plt.show()
else:
    print('Skipped: Q-tables not available.')

In [ ]:
# Q-table heatmaps: dominant secondary profile per state
if HAS_QTABLES and qtables is not None:
    plot_secondary_heatmaps(qtables, n_agents)
    plt.show()
else:
    print('Skipped: Q-tables not available.')

In [ ]:
# Top-5 Q-values per agent (works with saved Q-tables, no in-memory agents required)
if HAS_QTABLES and qtables is not None:
    a1_names = ActionProfileMapper.auction_profile_names()
    a2_names = ActionProfileMapper.secondary_profile_names()
    disc = StateDiscretizer()

    for i in range(n_agents):
        key = f'agent_{i}'
        if key not in qtables:
            print(f'\nA{i+1}: Q-table missing ({key})')
            continue

        qt = qtables[key]  # expected shape: [state, auction_profile, secondary_profile]
        k = 5
        flat = qt.reshape(-1)
        top_idx = np.argpartition(flat, -k)[-k:]
        top_idx = top_idx[np.argsort(flat[top_idx])[::-1]]

        print(f'\nA{i+1} ({archetypes[i]}) — Top 5 Q-values:')
        for idx in top_idx:
            s, a1, a2 = np.unravel_index(idx, qt.shape)
            qval = qt[s, a1, a2]
            bins = disc.index_to_bins(int(s))
            state_desc = ', '.join([['early','mid','late'][bins[0]],
                                    ['lowP','medP','highP'][bins[1]],
                                    ['dirty','mixed','green'][bins[2]],
                                    ['deficit','balanced','surplus'][bins[3]],
                                    ['noCF','someCF','heavyCF'][bins[4]]])
            print(f'  Q={qval:+.3f} | state=({state_desc}) | '
                  f'auction={a1_names[a1]} | secondary={a2_names[a2]}')
else:
    print('Skipped: Q-tables not available.')

## 8. Strategy Frequency Analysis

In [ ]:
# Strategy frequency in last 500 episodes
plot_strategy_frequency(ql_df, n_agents, last_n=500)
plt.show()

## 9. Greedy Evaluation (100 Episodes)

In [ ]:
if EVAL_AGENTS is not None and EVAL_CONFIG is not None:
    eval_results = evaluate_qlearning(
        EVAL_AGENTS, EVAL_CONFIG, seed=SEED,
        n_eval_episodes=100
    )
else:
    eval_results = None
    print('Skipped: evaluation requires trained agents or saved Q-tables.')

In [ ]:
# Evaluation results visualization
if eval_results is not None:
    fig, axes = plt.subplots(1, 4, figsize=(20, 4))

    # Rewards
    mean_rew = eval_results['rewards'].mean(axis=0)
    std_rew = eval_results['rewards'].std(axis=0)
    axes[0].bar(range(n_agents), mean_rew, yerr=std_rew, capsize=3,
                color=['steelblue' if i%2==0 else 'forestgreen' for i in range(n_agents)])
    axes[0].set_title('Eval: Mean Reward')
    axes[0].set_xticks(range(n_agents))
    axes[0].set_xticklabels([f'A{i+1}' for i in range(n_agents)])
    axes[0].grid(True, alpha=0.3)

    # Green fracs
    mean_gf = eval_results['green_fracs'].mean(axis=0)
    axes[1].bar(range(n_agents), mean_gf * 100,
                color=['steelblue' if i%2==0 else 'forestgreen' for i in range(n_agents)])
    axes[1].set_title('Eval: Final Green %')
    axes[1].set_xticks(range(n_agents))
    axes[1].set_xticklabels([f'A{i+1}' for i in range(n_agents)])
    axes[1].grid(True, alpha=0.3)

    # Compliance
    mean_comp = eval_results['compliance'].mean(axis=0)
    axes[2].bar(range(n_agents), mean_comp * 100,
                color=['steelblue' if i%2==0 else 'forestgreen' for i in range(n_agents)])
    axes[2].set_title('Eval: Compliance Rate %')
    axes[2].set_xticks(range(n_agents))
    axes[2].set_xticklabels([f'A{i+1}' for i in range(n_agents)])
    axes[2].set_ylim(0, 105)
    axes[2].grid(True, alpha=0.3)

    # Price distribution
    axes[3].hist(eval_results['prices'], bins=20, color='tab:red', alpha=0.7)
    axes[3].set_title('Eval: Final Clearing Price Distribution')
    axes[3].set_xlabel('EUR/t')
    axes[3].grid(True, alpha=0.3)

    plt.suptitle('Greedy Evaluation Results (100 Episodes)', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('Skipped: eval_results is not available.')

## 10. Comparison with PPO/HAPPO (if available)

Load PPO training logs and overlay the learning curves.

In [ ]:
# Try to load PPO results for comparison
ppo_dir = 'results/'
ppo_log_path = os.path.join(ppo_dir, f'training_log_s{SEED}.csv')

if os.path.exists(ppo_log_path):
    ppo_df = pd.read_csv(ppo_log_path)
    print(f'PPO log loaded: {len(ppo_df)} episodes')
    HAS_PPO = True
else:
    print(f'PPO log not found at {ppo_log_path} - comparison plots will show Q-learning only.')
    print('Run PPO training first, then re-run this cell.')
    ppo_df = None
    HAS_PPO = False

In [ ]:
# Reward comparison
plot_reward_comparison(ql_df, ppo_df, n_agents)
plt.show()

In [ ]:
# Price comparison
plot_price_comparison(ql_df, ppo_df)
plt.show()

In [ ]:
# Green fraction comparison
plot_green_comparison(ql_df, ppo_df, n_agents)
plt.show()

In [ ]:
# Compliance comparison
plot_compliance_comparison(ql_df, ppo_df, n_agents)
plt.show()

## 11. Single-seed summary statistics

In [ ]:
# Final summary table
last_500 = ql_df.tail(500)

summary_data = []
for i in range(n_agents):
    row = {
        'Agent': f'A{i+1}',
        'Archetype': archetypes[i],
        'Avg Reward': last_500[f'reward_A{i+1}'].astype(float).mean(),
        'Final Green %': last_500[f'green_frac_A{i+1}'].astype(float).mean() * 100,
        'Compliance %': last_500[f'compliance_rate_A{i+1}'].astype(float).mean() * 100,
        'Avg Shortfall': last_500[f'shortfall_A{i+1}'].astype(float).mean(),
    }
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
print('Q-Learning Performance (last 500 training episodes):')
print(summary_df.to_string(index=False, float_format='%.2f'))

if HAS_PPO:
    print('\n--- For comparison, PPO (last 500 episodes): ---')
    ppo_last = ppo_df.tail(500)
    for i in range(n_agents):
        col_r = f'reward_A{i+1}'
        col_g = f'green_frac_A{i+1}'
        if col_r in ppo_last.columns:
            ppo_rew = ppo_last[col_r].astype(float).mean()
            ppo_gf = ppo_last[col_g].astype(float).mean() * 100 if col_g in ppo_last.columns else 0
            print(f'  A{i+1}: reward={ppo_rew:.2f}, green={ppo_gf:.1f}%')

## 12. Sub-RQ 1 — Q-learning vs the default-config PPO sweep

**Sub-RQ 1 (simulation credibility).** *What environment and algorithm design choices are required to build a stable and behaviourally credible simulation of a carbon market?* The two operational claims we want to defend are:

* **RQ 1.a — convergence.** Agents converge to stable policies within a recognisable horizon.
* **RQ 1.b — reproducibility.** Findings replicate across seeds; cross-seed noise stays bounded.

Q-learning's role here is **as a floor**, not a contender. Tabular Q-learning runs in the *same* environment as PPO — same auction, same MSR, same warm-start — but compresses observations to 243 discrete states and the action space to 24 hand-coded profiles. If an environment is too simple, both algorithms succeed and the credibility of the PPO result is unclear. If the environment is too pathological, neither succeeds. The **gap** between Q-learning and PPO on the cross-seed metrics below is what tells us the simulation is non-trivial *and* solvable — i.e. credible.

Concretely, this section:
1. Loads the default-config PPO sweep (`results/sweeps/default_seeds/default/training_log_s*.csv`).
2. Loads all available Q-learning seeds (`results/qlearning/ql_training_log_s*.csv`).
3. Builds **cross-seed fan charts** for the four headline metrics: clearing price, compliance rate, mean    green fraction, and `quality_score`.
4. Reports a small **convergence-plateau** check (rolling-mean tolerance band, à la C1 in the data-science    plan).
5. Tabulates **converged-window** mean ± std for both algorithms side by side.


In [ ]:
# 12.1  Locate sweep + Q-learning logs
import glob, re

# Default-config PPO sweep — see configs/sweeps/default_seeds.yaml
SWEEP_DIR_CANDIDATES = [
    'results/sweeps/default_seeds/default',     # canonical sweep output
    'results/sweeps/default_seeds',             # fallback if variants are flattened
    'results',                                  # fallback: any training_log_s*.csv under results/
]

def _seed_from(path, prefix):
    m = re.search(rf'{re.escape(prefix)}_s(\d+)\.csv$', path.replace('\\', '/'))
    return int(m.group(1)) if m else None

ppo_logs = {}
for d in SWEEP_DIR_CANDIDATES:
    if not os.path.isdir(d):
        continue
    paths = sorted(glob.glob(os.path.join(d, 'training_log_s*.csv')))
    for p in paths:
        s = _seed_from(p, 'training_log')
        if s is not None and s not in ppo_logs:
            ppo_logs[s] = p
    if ppo_logs:
        break

ql_paths = sorted(glob.glob('results/qlearning/ql_training_log_s*.csv'))
ql_logs = {_seed_from(p, 'ql_training_log'): p for p in ql_paths
           if _seed_from(p, 'ql_training_log') is not None}

print(f'Q-learning seeds   ({len(ql_logs):2d}): {sorted(ql_logs)}')
print(f'PPO sweep seeds    ({len(ppo_logs):2d}): {sorted(ppo_logs)}')

HAS_SWEEP = len(ppo_logs) >= 2 and len(ql_logs) >= 1
if not HAS_SWEEP:
    print('\nSkipping cross-seed comparison.\n'
          'Run:  python scripts/sweep.py --spec configs/sweeps/default_seeds.yaml\n'
          'and:  python src/train_qlearning.py --seeds 42 123 456')


In [ ]:
# 12.2  Cross-seed loader — pulls the four credibility metrics into a tidy long frame
if HAS_SWEEP:
    PPO_METRICS = {
        'clearing_price':  ['ep_mean_clearing_price', 'clearing_price_last'],
        'compliance_rate': None,   # mean over agent compliance_rate_A* / shortfall_A*
        'green_frac':      None,   # mean over agent green_frac_A*
        'quality_score':   ['quality_score'],
    }

    def _agent_mean(df, prefix, n_total):
        cols = [f'{prefix}_A{i+1}' for i in range(n_total) if f'{prefix}_A{i+1}' in df.columns]
        if not cols:
            return None
        return df[cols].apply(pd.to_numeric, errors='coerce').mean(axis=1)

    def _ppo_compliance(df, n_total):
        cols = [f'compliance_rate_A{i+1}' for i in range(n_total) if f'compliance_rate_A{i+1}' in df.columns]
        if cols:
            return df[cols].apply(pd.to_numeric, errors='coerce').mean(axis=1)
        # Fallback: derive from shortfall (PPO log records shortfall, not compliance_rate, by default)
        sf = [f'shortfall_A{i+1}' for i in range(n_total) if f'shortfall_A{i+1}' in df.columns]
        if not sf:
            return None
        return (df[sf].apply(pd.to_numeric, errors='coerce') < 1e-6).mean(axis=1)

    def _series(df, metric, n_total):
        if metric == 'compliance_rate':
            return _ppo_compliance(df, n_total)
        if metric == 'green_frac':
            return _agent_mean(df, 'green_frac', n_total)
        for col in (PPO_METRICS[metric] or []):
            if col in df.columns:
                return pd.to_numeric(df[col], errors='coerce')
        return None

    n_total_cfg = config['companies']['n_agents'] + config['companies'].get('n_bot_agents', 0)

    def _stack(logs):
        """Return dict[metric] -> DataFrame with columns = seeds and rows = episode index."""
        out = {m: {} for m in PPO_METRICS}
        for seed, path in sorted(logs.items()):
            df = pd.read_csv(path)
            for m in PPO_METRICS:
                s = _series(df, m, n_total_cfg)
                if s is not None:
                    out[m][seed] = s.reset_index(drop=True)
        return {m: pd.DataFrame(d) for m, d in out.items() if d}

    ppo_stacks = _stack(ppo_logs)
    ql_stacks  = _stack(ql_logs)
    print({m: f'{ppo_stacks[m].shape} ppo / '
              f'{ql_stacks.get(m, pd.DataFrame()).shape} ql' for m in ppo_stacks})


In [ ]:
# 12.3  Fan charts: PPO mean ± 1 std band vs Q-learning mean across seeds
if HAS_SWEEP:
    def _smoothed(df, win):
        return df.rolling(win, min_periods=max(1, win // 5)).mean()

    fig, axes = plt.subplots(2, 2, figsize=(15, 9))
    axes = axes.ravel()
    metric_titles = {
        'clearing_price':  'Mean clearing price (EUR/t)',
        'compliance_rate': 'Compliance rate (mean across agents)',
        'green_frac':      'Green fraction (mean across agents)',
        'quality_score':   'quality_score  (anchor-invariant, signed)',
    }
    for ax, metric in zip(axes, metric_titles):
        ppo_df_m = ppo_stacks.get(metric)
        ql_df_m  = ql_stacks.get(metric)
        if ppo_df_m is None or ppo_df_m.empty:
            ax.set_axis_off(); ax.set_title(f'{metric_titles[metric]} (no data)'); continue
        win = max(50, len(ppo_df_m) // 100)
        ppo_smooth = _smoothed(ppo_df_m, win)
        ax.plot(ppo_smooth.index, ppo_smooth.mean(axis=1), color='tab:orange',
                lw=2, label=f'PPO mean  (n={ppo_df_m.shape[1]})')
        ax.fill_between(ppo_smooth.index,
                        ppo_smooth.mean(axis=1) - ppo_smooth.std(axis=1),
                        ppo_smooth.mean(axis=1) + ppo_smooth.std(axis=1),
                        color='tab:orange', alpha=0.2, label='PPO ± 1 std')
        if ql_df_m is not None and not ql_df_m.empty:
            ql_win = max(20, len(ql_df_m) // 100)
            ql_smooth = _smoothed(ql_df_m, ql_win)
            ax.plot(ql_smooth.index, ql_smooth.mean(axis=1), color='tab:blue',
                    lw=2, label=f'Q-learning mean  (n={ql_df_m.shape[1]})')
            ax.fill_between(ql_smooth.index,
                            ql_smooth.mean(axis=1) - ql_smooth.std(axis=1),
                            ql_smooth.mean(axis=1) + ql_smooth.std(axis=1),
                            color='tab:blue', alpha=0.18)
        if metric == 'quality_score':
            ax.axhline(0, color='grey', lw=0.5, ls='--')
        ax.set_title(metric_titles[metric]); ax.set_xlabel('Episode')
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.suptitle('Sub-RQ 1 — Q-learning vs default-sweep PPO/HAPPO  (cross-seed fan)',
                 fontsize=13)
    plt.tight_layout(); plt.show()


In [ ]:
# 12.4  Plateau detection — first episode where the rolling mean stays inside a
#       tolerance band for K consecutive windows. Mirrors the C1 method in
#       planning/data_science_analysis_plan.md (RQ 1.a).
if HAS_SWEEP:
    def first_plateau(series: pd.Series, win: int = 50, tol: float = 0.05,
                      k_windows: int = 4) -> int | None:
        """Return episode index of the first window whose rolling mean stays
        within ±tol·|level| of its median for k_windows consecutive windows;
        None if no plateau is reached."""
        s = pd.to_numeric(series, errors='coerce').rolling(win).mean().dropna()
        if len(s) < k_windows * win:
            return None
        for start in range(len(s) - k_windows * win):
            window_vals = s.iloc[start:start + k_windows * win]
            level = window_vals.median()
            band = max(abs(level), 1.0) * tol
            if (window_vals.max() - window_vals.min()) <= 2 * band:
                return int(s.index[start])
        return None

    rows = []
    for label, stack in [('Q-learning', ql_stacks), ('PPO sweep', ppo_stacks)]:
        for metric in ['clearing_price', 'quality_score', 'compliance_rate']:
            df = stack.get(metric)
            if df is None or df.empty:
                continue
            mean_series = df.mean(axis=1)
            ep = first_plateau(mean_series)
            rows.append({
                'algorithm': label, 'metric': metric,
                'plateau_episode': ep,
                'final_mean':  round(float(mean_series.tail(max(1, len(mean_series)//10)).mean()), 3),
                'final_std':   round(float(df.tail(max(1, len(df)//10)).std(axis=1).mean()), 3),
                'n_seeds':     int(df.shape[1]),
                'n_episodes':  int(df.shape[0]),
            })
    plateau_df = pd.DataFrame(rows)
    print('Convergence plateau (RQ 1.a) and cross-seed std (RQ 1.b)\n'
          + '-' * 72)
    print(plateau_df.to_string(index=False))


In [ ]:
# 12.5  Converged-window summary table (last 10% of episodes, mean ± std across seeds)
if HAS_SWEEP:
    def converged_summary(stack, frac=0.10):
        out = {}
        for metric, df in stack.items():
            tail = df.tail(max(1, int(len(df) * frac)))
            out[metric] = (tail.mean().mean(), tail.mean().std())
        return out

    ql_sum = converged_summary(ql_stacks)
    pp_sum = converged_summary(ppo_stacks)
    table = []
    for metric in ['clearing_price', 'compliance_rate', 'green_frac', 'quality_score']:
        ql_mu, ql_sd = ql_sum.get(metric, (float('nan'), float('nan')))
        pp_mu, pp_sd = pp_sum.get(metric, (float('nan'), float('nan')))
        table.append({
            'metric':            metric,
            'Q-learning  μ':     round(ql_mu, 3),
            'Q-learning  σ_seed': round(ql_sd, 3),
            'PPO sweep   μ':     round(pp_mu, 3),
            'PPO sweep   σ_seed': round(pp_sd, 3),
        })
    print('Converged-window mean ± cross-seed std  (last 10% of episodes)\n'
          + '-' * 78)
    print(pd.DataFrame(table).to_string(index=False))


---

## 13. Takeaways — Sub-RQ 1 lens

**On simulation credibility (Sub-RQ 1).** Q-learning is included here as a *floor*, not a contender. The comparison in §11 sits the tabular baseline directly next to the PPO/HAPPO default sweep on the same metrics, computed from the same `ETSEnvironment`:

* **Convergence (RQ 1.a).** Q-learning typically plateaus within a few thousand episodes — far earlier   than PPO — because its 243-state × 24-action search space saturates fast. PPO's plateau episode in   the same `quality_score` series is the reference for the credibility claim.
* **Reproducibility (RQ 1.b).** Cross-seed std on the converged window is reported in §11.5 for both   algorithms. Cross-seed noise that's small relative to the cross-algorithm gap is what licenses the   rest of the thesis.
* **Recognisability.** A converged Q-learning policy already produces an EU-ETS-shaped trajectory —   positive `quality_score`, anchor-tracking clearing price, plateaued green fraction. That tells us the   *environment* (auction + MSR + warm-start + reward) is the dominant driver of the qualitative   outcome; PPO's gain is in the magnitude of the optima, not their existence.

**Why a tabular baseline is the right baseline.** The discretisation deliberately throws away information that PPO has access to (continuous bid/qty/invest, opponent modelling, terminal-value shaping). Anything PPO does on top of Q-learning is therefore evidence that the extra state and policy expressivity *matters* for this market — which is exactly the design-choice question Sub-RQ 1 is asking.

**Out of scope for this notebook.** Cross-variant sensitivity (LRF / MSR / penalty sweeps) is Sub-RQ 2 territory and lives in the experiments notebook; strategy clustering and the financial-vs-ESG contrast are Sub-RQ 3 in `ets_marl - Default RQ Analysis.ipynb`.
